In [1]:
# import numpy as np
# import matplotlib.pyplot as plt

# from lammps import lammps, LMP_STYLE_GLOBAL,LMP_STYLE_ATOM, LMP_STYLE_LOCAL, LMP_TYPE_SCALAR
# from lammps import LMP_TYPE_VECTOR, LMP_TYPE_ARRAY, LMP_SIZE_VECTOR, LMP_SIZE_ROWS, LMP_SIZE_COLS
# from ctypes import c_double, c_int

# import scipy
# from scipy import optimize

In [2]:
import LACT
from LACT import *

In [3]:
import numpy as np
import matplotlib.pyplot as plt

from lammps import lammps, LMP_STYLE_ATOM, LMP_TYPE_VECTOR, LMP_TYPE_ARRAY

#from lammps import lammps, LMP_STYLE_GLOBAL,LMP_STYLE_ATOM, LMP_STYLE_LOCAL, LMP_TYPE_SCALAR
#from lammps import LMP_TYPE_VECTOR, LMP_TYPE_ARRAY, LMP_SIZE_VECTOR, LMP_SIZE_ROWS, LMP_SIZE_COLS
from ctypes import c_double, c_int

import scipy
from scipy import optimize

In [4]:
#lmp = lammps()
lmp = lammps(cmdargs=['-screen', 'none'])

lmp.commands_string('''
# ------------------------ INITIALIZATION ----------------------------
processors    * * *
units         metal
dimension    3
boundary    p    p    p
atom_style   atomic
atom_modify map yes


box        tilt large
#--------------------------- LAMMPS Data File--------------------------
#read_data    input_data/thicker_domain.lmp
read_data    input_data/4b_step.lmp
change_box    all triclinic
reset_atom_ids sort yes

#--------------
pair_style    eam/alloy
pair_coeff    * * input_data/Cu_mishin1.eam.alloy Cu

#-------------------Various continuation commands----------------------
compute forces all property/atom fx fy fz
compute ids all property/atom id
compute x_check all property/atom x y z
#atom_modify map yes
''')

natoms = lmp.extract_global("natoms")
original_box_size = lmp.extract_box()
ref_X = np.reshape(
            np.array(lmp.gather_atoms("x", 1, 3)),
            (natoms, 3),
        ).copy()

In [5]:
# #### helper functions

# ### periodicity preservation
# def fix_periodicity(X,box_size,show=False):
#     kk = 0
#     for ii in range(len(X)):
#         for j in range(3):
#             if X[ii][j] < box_size[0][j]:
#                 X[ii][j] += box_size[1][j] - box_size[0][j]
#                 kk+=1
#             elif X[ii][j] > box_size[1][j]:
#                 X[ii][0] -= box_size[1][j] - box_size[0][j]
#                 kk+=1
#     if show:
#         print("Number of changed coordinates:",kk)
    
# def fix_periodicity_flat(X,box_size,show=False):
#     kk = 0
#     for ii in range(len(X)):
#         j = ii % 3
#         if X[ii] < box_size[0][j]:
#             X[ii] += box_size[1][j] - box_size[0][j]
#             kk+=1
#         elif X[ii] > box_size[1][j]:
#             X[ii] -= box_size[1][j] - box_size[0][j]
#             kk+=1
#     if show:
#         print("Number of changed coordinates:",kk)
        
# def fix_periodicity_relative(X,box_size,show=False):
#     kk = 0
#     allowed_directions = (np.array(box_size[1]) - np.array(box_size[0]))/2
#     for ii in range(len(X)):
#         for j in range(3):
#             if X[ii][j] < -allowed_directions[j]:
#                 X[ii][j] += 2*allowed_directions[j]
#                 kk+=1
#             elif X[ii][j] > allowed_directions[j]:
#                 X[ii][0] -= 2*allowed_directions[j]
#                 kk+=1
#     if show:
#         print("Number of changed coordinates:",kk)
        
# def fix_periodicity_relative_flat(X,box_size,show=False):
#     kk = 0
#     allowed_directions = (np.array(box_size[1]) - np.array(box_size[0]))/2
#     for ii in range(len(X)):
#         j = ii % 3
#         if X[ii] < -allowed_directions[j]:
#             X[ii] += 2*allowed_directions[j]
#             kk+=1
#         elif X[ii] > allowed_directions[j]:
#             X[ii] -= 2*allowed_directions[j]
#             kk+=1
        
#     if show:
#         print("Number of changed coordinates:",kk)
        
# ### LAMMPS interface:

# # passing the flat dN+1 variable Y to LAMMPS
# def pass_X(Y):
#     lammps_commands = f"""
#      change_box all x final {original_box_size[0][0]+Y[-1]} {original_box_size[1][0]-Y[-1]} units box
#     #fix        relax all box/relax iso 0.0    
#     """
#     lmp.commands_string(lammps_commands)
#     box_size = lmp.extract_box()[:2]
#     Y_x = Y[0:-1]
#     fix_periodicity_flat(Y_x,box_size)
#     Y_x = ((len(Y_x))*c_double)(*Y_x)
#     lmp.scatter_atoms("x", 1, 3, Y_x)

# # extended system
# def G(Y,Y0,Ydot,ds,verbose = False):
#     pass_X(Y)
#     lmp.command('run 0')
#     f_t = lmp.numpy.extract_compute('forces',LMP_STYLE_ATOM,LMP_TYPE_ARRAY)
#     _IDS = lmp.numpy.extract_compute('ids',LMP_STYLE_ATOM,LMP_TYPE_VECTOR).astype('int32')
#     f_t = f_t[np.argsort(_IDS)]
#     G0 = f_t.flatten()
#     box_size = lmp.extract_box()[:2]
#     YminusY0 = Y-Y0
#     fix_periodicity_relative_flat(YminusY0[:-1],box_size)
#     last_eqn = (YminusY0*Ydot).sum() - ds
#     G0 = np.append(G0,last_eqn)
#     if verbose:
#         print(Y[-1])
#         print(last_eqn)
#         print(np.linalg.norm(G0,np.Inf))
#         #lmp.command('write_dump all custom run_dump2 id type x y z xu yu zu')
#     return G0

In [6]:
#energies
E = []
#optimal extended variables
Y = []


# Do we start it as new continuation run or continue one from before?
new_start = True

if new_start:
    #populate them with two consecutive data points from previous runs:
    ii = 44
    forward = True
    rr = range(2) if forward else reversed(range(1))

    for k in rr:
        lmp.command(f'read_dump dumps/thin_domain/minimise_dump/minimise_dump_{ii+k} 0 x y z box yes')
        lmp.command('run 0')
        #_E = lmp.get_thermo("pe")
        #_max_force = lmp.get_thermo("fmax")
        #_IDS = np.array(lmp.gather_atoms("id", 0, 1))
        #_IDS = lmp.numpy.extract_compute('ids',LMP_STYLE_ATOM,LMP_TYPE_VECTOR).astype('int32')
        #_F = lmp.numpy.extract_compute('forces',LMP_STYLE_ATOM,LMP_TYPE_ARRAY).copy()
        _X = np.reshape(
                 np.array(lmp.gather_atoms("x", 1, 3)),
                 (natoms, 3),
             )
        _μ = -(original_box_size[0][0] - lmp.extract_box()[0][0])
        _Y= np.append(_X.flatten(),_μ)
        Y += [_Y]
        lmp.command(f'write_dump all custom dumps/thin_domain/continuation_dumps/default_branch/dump_{len(Y)} id type x y z xu yu zu')  
        
else:
    for k in range(1348): 
        lmp.command(f'read_dump dumps/thin_domain/continuation_dumps/default_branch/dump_{k+1} 0 x y z box yes')
        lmp.command('run 0')
        _X = np.reshape(
                 np.array(lmp.gather_atoms("x", 1, 3)),
                 (natoms, 3),
             )
        _μ = -(original_box_size[0][0] - lmp.extract_box()[0][0])
        _Y= np.append(_X.flatten(),_μ)
        Y += [_Y]
        lmp.command('reset_timestep 0') 

In [7]:
len(Y)

2

In [8]:
def continuation_step(Y,ds, lmp,original_box_size):
    pass_ext_variable_info(Y[-1],lmp,original_box_size)
    box_size = lmp.extract_box()[:2]
    Y_dot = Y[-1] - Y[-2]
    fix_periodicity_relative_flat(Y_dot[:-1],box_size)
    Y_dot = Y_dot / np.linalg.norm(Y_dot)
    Y_0 = Y[-1] + ds*Y_dot
    
    Y_1 = scipy.optimize.root(extended_system, Y_0,
                   args=(Y[-1],Y_dot,ds,lmp,original_box_size),
#                    tol = 1e-4,
                   method='krylov',
                   options={'disp': True,
                            'fatol': 1e-4,
                            'maxiter': 6,
                            'line_search': 'armijo' ,
                            'jac_options': {'inner_M': None, 
                                            'method': 'lgmres'}},
                   callback=None)
    return Y_1

In [9]:
ds_default = 1e-2
ds_smallest = 1e-3
ds_largest = 2e0
ds = ds_default
counter = 0

In [10]:
for k in range(5):
    Y_1 = continuation_step(Y,ds,lmp,original_box_size)
    if Y_1.success == False:
        ds = ds/2
        if ds > ds_smallest:
            print("ds halved, now equal to ", ds, ". We go 2 steps back.")
        else:
            print("ds now below the threshold, equal to ", ds, ".  Aborting...")
            break
        res_at = -2
        Y = Y[:res_at]
        counter = 0
    else:
        Y += [Y_1.x]
        pass_ext_variable_info(Y[-1],lmp,original_box_size)
        lmp.command('reset_timestep 0')
        lmp.command(f'write_dump all custom dumps/thin_domain/continuation_dumps/default_branch/dump_{len(Y)} id type x y z xu yu zu')        
        counter += 1
        if counter > 5 and ds < ds_largest:
            ds = 2*ds
            print("ds doubled, now equal to ", ds, ".")
            counter = 0
        
        print("Iteration step: ",k," ",", Solution step: ",len(Y)," ",", Continuation parameter: ", Y[-1][-1])
    if np.sign(Y[-1][-1] - Y[-2][-1])*np.sign(Y[-2][-1] - Y[-3][-1]) < 0.0:
        print("turn, turn, turn")
    

0:  |F(x)| = 5.70745e-06; step 1
Iteration step:  0   , Solution step:  3   , Continuation parameter:  4.400245129696467
0:  |F(x)| = 2.07827e-05; step 1
Iteration step:  1   , Solution step:  4   , Continuation parameter:  4.401068295436362


/home/mb/.local/lib/python3.11/site-packages/scipy/optimize/_nonlin.py:367: RuntimeWarning: invalid value encountered in scalar divide
  and dx_norm/self.x_rtol <= x_norm))


0:  |F(x)| = 5.91525e-06; step 1
Iteration step:  2   , Solution step:  5   , Continuation parameter:  4.401188909792215
0:  |F(x)| = 4.7757e-06; step 1
Iteration step:  3   , Solution step:  6   , Continuation parameter:  4.401433288754458
0:  |F(x)| = 1.22667e-06; step 1
Iteration step:  4   , Solution step:  7   , Continuation parameter:  4.401662702392054


In [146]:
Y[1]

array([-59.5877 ,   3.31059, 401.707  , ...,   1.87307, 472.471  ,
         4.4    ])

In [11]:
# ii = -2

# pass_X(Y[ii-2])
# box_size = lmp.extract_box()[:2]
# Y_dot = Y[ii-2] - Y[ii-3]
# fix_periodicity_relative_flat(Y_dot[:-1],box_size)
# Y_dot = Y_dot / np.linalg.norm(Y_dot)

# np.linalg.norm(G(Y[ii-1],Y[ii-2],Y_dot,0,verbose=True)[:-1],np.Inf)

In [18]:
a = "x"
b = "y"
c = f"{b}"
print(a+b+c)

xyy


In [12]:
original_box_size[0][0]

-64.3409885018

'\n     change_box all x final 0 1 units box\n    '

In [24]:
class SimpleScalarProblem:
    def inner(self, a, b):
        """The inner product in the problem domain. For scalars, this is just a
        multiplication.
        """
        return a * b

    def norm2_r(self, a):
        """The norm in the range space; used to determine if a solution has been found."""
        return a**2

    def f(self, u, lmbda):
        """The evaluation of the function to be solved"""
        return math.sin(u) - lmbda

    def df_dlmbda(self, u, lmbda):
        """The function's derivative with respect to the parameter. Used in Euler-Newton
        continuation.
        """
        return -1.0

    def jacobian_solver(self, u, lmbda, rhs):
        """A solver for the Jacobian problem. For scalars, this is just a division."""
        return rhs / math.cos(u)


problem = SimpleScalarProblem()

In [30]:
import math

In [39]:
?problem.norm2_r

Signature: problem.norm2_r(a)
Docstring: The norm in the range space; used to determine if a solution has been found.
File:      /tmp/ipykernel_129872/2140993704.py
Type:      method


In [37]:
problem.f(1,1)

-0.1585290151921035

In [73]:
my_print = lambda b : print("wtf1",b) 
#print(add_numbers("haha")) # 5 
my_print("lolz")

wtf1 lolz


In [131]:
class mynumber:
    wtf2 = [5,6]
    wtf3 = 11
    
    def __init__(self, value,function):
        self.wtf = value
        def ff(x):
            function(x)
            print("test")
        self.fff = ff

    def print_value(self):
        print(self.wtf)

obj1 = mynumber(18,my_print)
obj1.print_value()

obj2 = mynumber(17,print)

18


In [132]:
obj1.wtf

18

In [133]:
obj2.wtf

17

In [134]:
obj1.wtf2

[5, 6]

In [135]:
obj2.wtf2

[5, 6]

In [136]:
obj2.wtf2[0] = 10

In [137]:
obj1.wtf2

[10, 6]

In [138]:
obj2.wtf2

[10, 6]

In [139]:
obj1.wtf3

11

In [140]:
obj2.wtf3

11

In [141]:
obj1.wtf3 = 12

In [142]:
obj1.wtf3

12

In [143]:
obj2.wtf3

11

In [56]:
def fff(x): x^2

In [42]:
obj1.wtf2

5

In [36]:
obj1.wtf

17

In [95]:
class atom_cont_system:
    def __init__(self,lmp,update_command):
        self.lmp = lmp
        self.org_box = lmp.extract_box()
        self.ref_X = np.reshape(
            np.array(lmp.gather_atoms("x", 1, 3)),
            (natoms, 3),
        ).copy()
        self.change_cont_param = lambda x : update_command(x,self.org_box)
         
#    def pass_ext_variable_info(self,

In [97]:
def change_box_command(limits,direction="x"):
    command = f"""
     change_box all {direction} final {limits[0]} {limits[1]} units box
    """
    return command

def update_command_box_x(x,box_size):
    direction = "x"
    i = 0 if direction=="x" else 1 if direction=="y" else 2
    return change_box_command([box_size[0][i]+x,box_size[1][i]-x],direction=direction)

In [98]:
system = atom_cont_system(lmp,update_command_box_x)

In [100]:
system.change_cont_param(0.0)

'\n     change_box all x final -59.939325799407946 102.95835764760794 units box\n    '

In [101]:
original_box_size

([-64.3409885018, 0.0, 380.0],
 [107.36002035, 4.427452710080594, 500.0],
 0.0,
 0.0,
 0.0,
 [1, 1, 1],
 0)

In [94]:
update_command_box(0.02,original_box_size)

'\n     change_box all x final -64.3209885018 107.34002035 units box\n    '